<span style="font-family:monospace;letter-spacing:.14em;color:#857259">BANA · MODULE 2 · EXPLORING AND VISUALISING DATA</span>

# Session 2: Describing numbers: centre, spread and shape

**Dataset:** EV purchase intention (Kaggle Playground Series S6E9, a *synthetic* dataset). You work with a 10,000-row seeded sample, `ev_m2_sample.csv`.
**Time:** 60 minutes. **Course outcomes:** CO2 (and CO3 for Session 5).

| Block | Min | What you do |
|---|---|---|
| Hook | 5 | Anscombe's quartet: same numbers, different data |
| Concept | 10 | Mean vs median, spread, skew (instructor) |
| Guided lab | 30 | Profile age, income, commute; compare buyers and non-buyers |
| Team task | 10 | Defend one 'typical customer' number |
| Exit ticket | 5 | Three quick checks |

**By the end of this session you can:**
- choose between mean and median and defend the choice
- read a histogram for centre, spread and shape
- spot a suspicious spike
- compare a number across groups

**New commands this session (max 5):** `describe()`, `mean()` / `median()`, `sns.histplot`, `groupby().mean()`, `sort_values()`

**AI-use (AMBER protocol).** You may use an AI tool to look up *syntax*. The reasoning and interpretation must be your own: you will defend them in the viva.

In [ ]:
# -- Setup: run once ------------------------------------------------------
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter
warnings.filterwarnings("ignore")

# House style: Nikhil D Jonathan design system (sand ground, terracotta / lagoon / ochre data colours)
C = {"lagoon": "#25707B", "terracotta": "#B5562F", "ochre": "#E0A231", "moss": "#3F6F4B",
     "earth": "#665643", "ink": "#221B15", "sand": "#F8F3EA", "card": "#FFFDF9", "line": "#E3D6C1"}
sns.set_theme(style="whitegrid", rc={
    "figure.facecolor": C["sand"], "axes.facecolor": C["card"], "axes.edgecolor": "#CDBCA3",
    "grid.color": C["line"], "text.color": C["ink"], "axes.labelcolor": C["ink"],
    "xtick.color": "#4A3E31", "ytick.color": "#4A3E31", "axes.spines.top": False,
    "axes.spines.right": False, "figure.dpi": 100, "axes.titleweight": "bold",
    "axes.titlesize": 12, "axes.titlelocation": "left"})
PCT = PercentFormatter(1, decimals=0)                              # axis labels as percentages
SEQ = sns.light_palette(C["terracotta"], as_cmap=True)   # one-hue sequential map for heatmaps

# -- Load the data (Colab: upload ev_m2_sample.csv to the Files pane, or run the upload below)
FILE = "ev_m2_sample.csv"
if not os.path.exists(FILE):
    try:
        from google.colab import files
        print("Choose ev_m2_sample.csv"); files.upload()
    except ImportError:
        raise FileNotFoundError("Put ev_m2_sample.csv in the same folder as this notebook.")
raw = pd.read_csv(FILE)
df = raw.copy()
print("Loaded", df.shape[0], "rows x", df.shape[1], "columns")

## Part A. Hook: same numbers, different data (5 min)

In [ ]:
an = sns.load_dataset("anscombe")      # needs internet (Colab is fine)
print(an.groupby("dataset").agg(x_mean=("x", "mean"), y_mean=("y", "mean"), y_std=("y", "std")).round(2))
print("Correlation x-y:", {k: round(g["x"].corr(g["y"]), 3) for k, g in an.groupby("dataset")})
sns.lmplot(data=an, x="x", y="y", col="dataset", col_wrap=4, height=2.5, ci=None,
           scatter_kws={"color": C["terracotta"], "s": 28}, line_kws={"color": C["lagoon"]});

**Question:** the table says the four datasets are the same. What does the picture say? Write one sentence.

> _write here_

## Part B. Setup for this session
Run the setup cell above, then rebuild the derived columns.

In [ ]:
def add_derived(d):
    """The four derived columns built in Session 1 (pre-written so every notebook stands alone)."""
    d = d.copy()
    d["buy"] = (d["Will_Buy_EV"] == "Yes").astype(int)
    d["age_band"] = pd.cut(d["Age"], bins=[24, 34, 44, 54, 69], labels=["25-34", "35-44", "45-54", "55-69"])
    d["income_band"] = pd.qcut(d["Annual_Income_USD"], 4, labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"])
    d["charger_total"] = d["Charging_Stations_Near_Home"] + d["Charging_Stations_Near_Work"]
    return d

In [ ]:
df = add_derived(raw)
num_cols = ['Age', 'Annual_Income_USD', 'Daily_Commute_km']
df[num_cols].head()

## Part C. Describe the numbers (10 min)
**Predict first:** which of the three columns will have the biggest gap between mean and median? Write a guess.

In [ ]:
df[num_cols].describe().round(1)

In [ ]:
summary = pd.DataFrame({
    "mean": df[num_cols].mean(),
    "median": df[num_cols].____(),     # TODO: which method gives the middle value?
    "std": df[num_cols].std(),
}).round(1)
summary["mean_minus_median"] = (summary["mean"] - summary["median"]).round(1)
summary

**YOUR ANSWER:** Was your guess right? Which column has the biggest gap, and in which direction does the tail pull the mean?

> _write here_

## Part D. Look at the shape (10 min)
**Predict first:** sketch what you expect each histogram to look like (bell? flat? skewed?). Then run.

In [ ]:
widths = {"Age": 1, "Annual_Income_USD": ____, "Daily_Commute_km": ____}      # TODO: try 1000, then 3000, then 10000 for income. What changes?
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, col, colour in zip(axes, num_cols, [C["lagoon"], C["terracotta"], C["ochre"]]):
    sns.histplot(df[col], binwidth=widths[col], color=colour, ax=ax)
    ax.axvline(df[col].mean(), color=C["ink"], ls="--", lw=1.2)     # dashed = mean
    ax.axvline(df[col].median(), color=C["ink"], ls=":", lw=1.4)    # dotted = median
    ax.set_title(col.replace("_", " "))
    ax.set_xlabel("")
fig.suptitle("Dashed line = mean, dotted line = median", x=0.01, ha="left", fontsize=10)
plt.tight_layout()

**YOUR ANSWER:** Two of the three histograms have something odd at one end. Which, and what do you think it means?

> _write here_

In [ ]:
print(f"Share of rows with income exactly 30,000: {(df['Annual_Income_USD'] == 30000).mean():.1%}")
print(f"Share of rows with commute exactly 5 km:      {(df['Daily_Commute_km'] == 5).mean():.1%}")
avg = df["Daily_Commute_km"].mean()
near = ((df["Daily_Commute_km"] - avg).abs() <= 5).mean()
print(f"Share of people within 5 km of the 'average' commute of {avg:.0f} km: {near:.1%}")

## Part E. Compare buyers and non-buyers (10 min)

In [ ]:
by_target = df.groupby("____")[num_cols].mean().round(0)    # TODO: which column splits buyers from non-buyers?
by_target

In [ ]:
df.groupby("City_Type")["buy"].mean().____(ascending=False).round(3)    # TODO: sort the result from biggest to smallest

**YOUR ANSWER:** Which of age, income and commute separates buyers from non-buyers most? Which barely matters? (Look at the gap between the two rows.)

> _write here_

## Team task: defend one 'typical customer' number (10 min)
A manager asks: *What does a typical EV prospect look like?* In your pair, pick **one** of income, age or commute, choose **mean or median**, and write a one-sentence answer a manager can use. Then add one sentence on what the number hides.

In [ ]:
# Save your numeric profile (this is your Session 2 output)
profile = summary.copy()
profile.to_csv("numeric_profile.csv")
profile

## Exit ticket (5 min)
1. When would you report the median instead of the mean?
2. A histogram shows a tall bar at exactly one value. Name one real-world reason this might happen.
3. Age is the same for buyers and non-buyers. Does that prove age does not matter in real life? Why not?